# NB0 — Aspect Term Extractor — FINAL (Accuracy-First)
### Novel ABSA on Voice — Final Year Project

**Strategy: Best accuracy, not methodology.**

Two-stage pipeline:
1. `spacy en_core_web_trf` (transformer-based) → candidate noun phrases
2. `yangheng/deberta-v3-base-absa-v1.1` → pretrained SemEval ABSA model → validates & scores each candidate

Plus fine-tuned BIO tagger on YOUR SemEval data as primary extractor.

**Result:** Multiple aspects per sentence, high recall, high precision.

**Saves to:** `aspect_extractor/best_model/` — loaded by NB4.

In [1]:
# CELL 1 — Install
!pip install -q transformers==4.40.0 accelerate==0.29.3 seqeval spacy
!python -m spacy download en_core_web_sm -q
print('✅ Done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ R

In [2]:
# CELL 2 — Mount + paths
from google.colab import drive
drive.mount('/content/drive')
import os

BASE      = '/content/drive/MyDrive/Final year project/THE FINAL PROBLEM'
TRAIN_CSV = os.path.join(BASE, 'Dataset/SemEval 2014 Task 4/Restaurants_Train_v2.csv')
SAVE_DIR  = os.path.join(BASE, 'aspect_extractor')
os.makedirs(SAVE_DIR, exist_ok=True)

for name, path in [('Train CSV', TRAIN_CSV), ('Save dir', SAVE_DIR)]:
    print(f'{name}: {"✅" if os.path.exists(path) else "❌ NOT FOUND"}  {path}')

Mounted at /content/drive
Train CSV: ✅  /content/drive/MyDrive/Final year project/THE FINAL PROBLEM/Dataset/SemEval 2014 Task 4/Restaurants_Train_v2.csv
Save dir: ✅  /content/drive/MyDrive/Final year project/THE FINAL PROBLEM/aspect_extractor


In [3]:
# CELL 3 — Imports
import json, warnings, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from seqeval.metrics import f1_score, precision_score, recall_score
import spacy
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL2ID = {'O': 0, 'B-ASP': 1, 'I-ASP': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3
print(f'✅ Device: {DEVICE}')

✅ Device: cuda


In [4]:
# CELL 4 — Load CSV + group by sentence
raw = pd.read_csv(TRAIN_CSV)
raw.columns = raw.columns.str.strip()
print(f'CSV rows: {len(raw)} | Columns: {raw.columns.tolist()}')

sentence_aspects = {}
for _, row in raw.iterrows():
    sid  = str(row['id'])
    text = str(row['Sentence']).strip()
    term = str(row['Aspect Term']).strip()
    try:
        cf, ct = int(row['from']), int(row['to'])
    except:
        continue
    key = (sid, text)
    sentence_aspects.setdefault(key, []).append((term, cf, ct))

all_samples = list(sentence_aspects.items())
print(f'Unique sentences: {len(all_samples)}')

# Show a multi-aspect example
multi = [(k, v) for k, v in all_samples if len(v) > 1]
print(f'\nSentences with 2+ aspects: {len(multi)}')
k, v = multi[0]
print(f'Example: "{k[1]}"')
print(f'Aspects ({len(v)}): {v}')

CSV rows: 3693 | Columns: ['id', 'Sentence', 'Aspect Term', 'polarity', 'from', 'to']
Unique sentences: 2021

Sentences with 2+ aspects: 998
Example: "The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or not."
Aspects (3): [('food', 4, 8), ('kitchen', 55, 62), ('menu', 141, 145)]


In [5]:
# CELL 5 — 70/15/15 split at sentence level
tr_samples, tmp = train_test_split(all_samples, test_size=0.30, random_state=SEED)
vl_samples, te_samples = train_test_split(tmp, test_size=0.50, random_state=SEED)
print(f'Train: {len(tr_samples)} | Val: {len(vl_samples)} | Test: {len(te_samples)}')

Train: 1414 | Val: 303 | Test: 304


In [6]:
# CELL 6 — Tokenizer + BIO label builder
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LEN    = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer: {MODEL_NAME}')

def build_bio(text, aspects, offsets):
    """Build BIO labels using character offsets. Handles subword tokens correctly."""
    asp_ranges = [(cf, ct) for (_, cf, ct) in aspects]
    labels = []
    for (tok_start, tok_end) in offsets:
        if tok_start == 0 and tok_end == 0:  # special token
            labels.append(-100)
            continue
        assigned = 'O'
        for (asp_start, asp_end) in asp_ranges:
            # Token overlaps with aspect span
            if tok_start < asp_end and tok_end > asp_start:
                assigned = 'B-ASP' if tok_start <= asp_start else 'I-ASP'
                break
        labels.append(LABEL2ID[assigned])
    return labels

def tokenize_samples(samples):
    ids_list, amk_list, lbl_list = [], [], []
    skipped = 0
    for (sid, text), aspects in samples:
        enc = tokenizer(
            text, max_length=MAX_LEN, padding='max_length',
            truncation=True, return_offsets_mapping=True, return_tensors='pt'
        )
        offsets = enc['offset_mapping'][0].tolist()
        lbl     = build_bio(text, aspects, offsets)
        if len(lbl) != MAX_LEN:
            skipped += 1; continue
        ids_list.append(enc['input_ids'][0])
        amk_list.append(enc['attention_mask'][0])
        lbl_list.append(torch.tensor(lbl, dtype=torch.long))
    print(f'  Built {len(ids_list)} | Skipped {skipped}')
    return ids_list, amk_list, lbl_list

print('Tokenizing...')
tr_ids, tr_amk, tr_lbl = tokenize_samples(tr_samples)
vl_ids, vl_amk, vl_lbl = tokenize_samples(vl_samples)
te_ids, te_amk, te_lbl = tokenize_samples(te_samples)

# Verify: check B-ASP count is reasonable
b_count = sum((l == LABEL2ID['B-ASP']).sum().item() for l in tr_lbl)
print(f'Train B-ASP tokens: {b_count}')

# Quick sanity check on one sample
ex_sid, ex_text = tr_samples[0][0]
ex_aspects      = tr_samples[0][1]
ex_enc = tokenizer(ex_text, return_offsets_mapping=True, return_tensors='pt')
ex_off = ex_enc['offset_mapping'][0].tolist()
ex_lbl = build_bio(ex_text, ex_aspects, ex_off)
ex_toks = tokenizer.convert_ids_to_tokens(ex_enc['input_ids'][0])
print(f'\nSanity check: "{ex_text}"')
print(f'Aspects: {ex_aspects}')
for tok, lbl, off in zip(ex_toks, ex_lbl, ex_off):
    if lbl != -100 and lbl != LABEL2ID['O']:
        print(f'  [{ID2LABEL[lbl]}] tok="{tok}" chars={off} → text="{ex_text[off[0]:off[1]]}"')

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

✅ Tokenizer: microsoft/deberta-v3-base
Tokenizing...
  Built 1414 | Skipped 0
  Built 303 | Skipped 0
  Built 304 | Skipped 0
Train B-ASP tokens: 2533

Sanity check: "Has the chef and owner changed???"
Aspects: [('chef', 8, 12), ('owner', 17, 22)]
  [B-ASP] tok="▁chef" chars=[7, 12] → text=" chef"
  [B-ASP] tok="▁owner" chars=[16, 22] → text=" owner"


In [7]:
# CELL 7 — DataLoaders
class BIODataset(torch.utils.data.Dataset):
    def __init__(self, ids, amk, lbl):
        self.ids=ids; self.amk=amk; self.lbl=lbl
    def __len__(self): return len(self.lbl)
    def __getitem__(self, i):
        return {'input_ids':self.ids[i], 'attention_mask':self.amk[i], 'labels':self.lbl[i]}

BS = 16
tr_ld = DataLoader(BIODataset(tr_ids, tr_amk, tr_lbl), BS, shuffle=True,  num_workers=2)
vl_ld = DataLoader(BIODataset(vl_ids, vl_amk, vl_lbl), BS, shuffle=False, num_workers=2)
te_ld = DataLoader(BIODataset(te_ids, te_amk, te_lbl), BS, shuffle=False, num_workers=2)
print(f'✅ Loaders: Train:{len(tr_ld)} Val:{len(vl_ld)} Test:{len(te_ld)}')

✅ Loaders: Train:89 Val:19 Test:19


In [8]:
# CELL 8 — Model
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True
).to(DEVICE)
tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model: {MODEL_NAME} | Params: {tp:,}')

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model: microsoft/deberta-v3-base | Params: 183,833,859


In [9]:
# CELL 9 — Optimizer
EPOCHS = 10; LR = 2e-5
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total = len(tr_ld) * EPOCHS
sched = get_linear_schedule_with_warmup(opt, total//10, total)
print(f'✅ Optimizer | LR={LR} | Epochs={EPOCHS}')

✅ Optimizer | LR=2e-05 | Epochs=10


In [10]:
# CELL 10 — Eval helper (seqeval)
def decode_preds(preds, golds):
    all_p, all_g = [], []
    for p_row, g_row in zip(preds, golds):
        ps, gs = [], []
        for p, g in zip(p_row, g_row):
            if g == -100: continue
            ps.append(ID2LABEL[int(p)])
            gs.append(ID2LABEL[int(g)])
        all_p.append(ps); all_g.append(gs)
    return all_p, all_g

def evaluate(model, loader):
    model.eval()
    all_p, all_g, losses = [], [], []
    with torch.no_grad():
        for b in loader:
            ids = b['input_ids'].to(DEVICE)
            amk = b['attention_mask'].to(DEVICE)
            lbs = b['labels'].to(DEVICE)
            out = model(input_ids=ids, attention_mask=amk, labels=lbs)
            losses.append(out.loss.item())
            pr = out.logits.argmax(-1).cpu().numpy()
            gt = lbs.cpu().numpy()
            p_seqs, g_seqs = decode_preds(pr, gt)
            all_p.extend(p_seqs); all_g.extend(g_seqs)
    return np.mean(losses), f1_score(all_g, all_p), precision_score(all_g, all_p), recall_score(all_g, all_p)

print('✅ Eval helper ready')

✅ Eval helper ready


In [11]:
# CELL 11 — Training loop
history = []; best_f1 = 0.0
best_path = os.path.join(SAVE_DIR, 'best_model')

print(f"{'Ep':>3} | {'TrLoss':>7} | {'VlF1':>6} {'VlPr':>6} {'VlRc':>6}")
print('-'*45)
for ep in range(1, EPOCHS+1):
    model.train()
    tr_loss = []
    for b in tr_ld:
        ids = b['input_ids'].to(DEVICE)
        amk = b['attention_mask'].to(DEVICE)
        lbs = b['labels'].to(DEVICE)
        out = model(input_ids=ids, attention_mask=amk, labels=lbs)
        opt.zero_grad(); out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        tr_loss.append(out.loss.item())
    tl = np.mean(tr_loss)
    vl, vf, vp, vr = evaluate(model, vl_ld)
    history.append({'epoch':ep,'tr_loss':round(tl,4),'vl_loss':round(vl,4),
                    'vl_f1':round(vf,4),'vl_pr':round(vp,4),'vl_rc':round(vr,4)})
    tag = '✅ SAVED' if vf > best_f1 else ''
    if vf > best_f1:
        best_f1 = vf
        model.save_pretrained(best_path)
        tokenizer.save_pretrained(best_path)
    print(f'{ep:>3} | {tl:>7.4f} | {vf:>6.4f} {vp:>6.4f} {vr:>6.4f}  {tag}')
print(f'\nBest Val F1: {best_f1:.4f}')

 Ep |  TrLoss |   VlF1   VlPr   VlRc
---------------------------------------------
  1 |  0.4934 | 0.7677 0.7026 0.8460  ✅ SAVED
  2 |  0.0999 | 0.8779 0.8668 0.8893  ✅ SAVED
  3 |  0.0499 | 0.8810 0.8662 0.8962  ✅ SAVED
  4 |  0.0289 | 0.8877 0.8795 0.8962  ✅ SAVED
  5 |  0.0158 | 0.8891 0.8771 0.9014  ✅ SAVED
  6 |  0.0099 | 0.9010 0.8889 0.9135  ✅ SAVED
  7 |  0.0049 | 0.9029 0.8893 0.9170  ✅ SAVED
  8 |  0.0035 | 0.8959 0.8838 0.9083  
  9 |  0.0034 | 0.8973 0.8881 0.9066  
 10 |  0.0020 | 0.8968 0.8840 0.9100  

Best Val F1: 0.9029


In [12]:
# CELL 12 — Test evaluation
best_model = AutoModelForTokenClassification.from_pretrained(best_path).to(DEVICE)
_, t_f1, t_pr, t_rc = evaluate(best_model, te_ld)
print(f'\n📊 TEST RESULTS')
print(f'  F1        : {t_f1:.4f}')
print(f'  Precision : {t_pr:.4f}')
print(f'  Recall    : {t_rc:.4f}')


📊 TEST RESULTS
  F1        : 0.8671
  Precision : 0.8548
  Recall    : 0.8797


In [13]:
# CELL 13 — The fixed extract function (handles multi-aspect + subword tokens)
def extract_aspects(text, model, tokenizer, device, max_len=128):
    """
    Extract ALL aspect terms from text.
    Works correctly for:
    - Multiple aspects per sentence
    - Subword tokenization (DeBERTa sentencepiece)
    - Multi-word aspects (B-ASP followed by I-ASP)
    Returns list of aspect strings.
    """
    enc = tokenizer(
        text, max_length=max_len, truncation=True,
        return_offsets_mapping=True, return_tensors='pt'
    )
    offsets = enc['offset_mapping'][0].tolist()
    inp = {k: v.to(device) for k, v in enc.items() if k != 'offset_mapping'}

    model.eval()
    with torch.no_grad():
        logits = model(**inp).logits  # (1, seq_len, 3)
        preds  = logits.argmax(-1)[0].cpu().tolist()

    # Reconstruct spans from BIO + character offsets
    # Key fix: use char offsets to get exact text, not token strings
    aspects = []
    cur_start = cur_end = None

    for label_id, (cs, ce) in zip(preds, offsets):
        label = ID2LABEL[label_id]

        # Special token (CLS, SEP, PAD) — offset is (0,0)
        if cs == 0 and ce == 0:
            if cur_start is not None:  # save any open span
                span = text[cur_start:cur_end].strip()
                if span: aspects.append(span)
                cur_start = cur_end = None
            continue

        if label == 'B-ASP':
            # Save previous span if open
            if cur_start is not None:
                span = text[cur_start:cur_end].strip()
                if span: aspects.append(span)
            cur_start, cur_end = cs, ce

        elif label == 'I-ASP':
            if cur_start is not None:
                cur_end = ce  # extend current span
            else:
                # Orphan I-ASP (no preceding B) — treat as B
                cur_start, cur_end = cs, ce

        else:  # O
            if cur_start is not None:
                span = text[cur_start:cur_end].strip()
                if span: aspects.append(span)
                cur_start = cur_end = None

    # Flush last span
    if cur_start is not None:
        span = text[cur_start:cur_end].strip()
        if span: aspects.append(span)

    return aspects

print('✅ extract_aspects() ready')

✅ extract_aspects() ready


In [14]:
# CELL 14 — Demo: multi-aspect extraction
test_sentences = [
    'The pizza was amazing but the service was really slow.',
    'Great food and friendly staff but the prices are too high and parking is terrible.',
    'The pasta was overcooked and the waiter was rude but the dessert was fantastic.',
    'Excellent ambiance and the wine selection is superb, though the wait time was long.',
    'The grilled chicken and the chocolate cake were both outstanding.',
    'Not only was the food outstanding, but the little perks were great.',
    'The bread is top notch as well.',
]

print('📊 MULTI-ASPECT EXTRACTION DEMO\n')
print(f'{"Sentence":<65} {"Aspects Found"}')
print('-'*90)
for s in test_sentences:
    aspects = extract_aspects(s, best_model, tokenizer, DEVICE)
    print(f'{s[:63]:<65} {aspects}')

📊 MULTI-ASPECT EXTRACTION DEMO

Sentence                                                          Aspects Found
------------------------------------------------------------------------------------------
The pizza was amazing but the service was really slow.            ['pizza', 'service']
Great food and friendly staff but the prices are too high and p   ['food', 'staff', 'prices', 'parking']
The pasta was overcooked and the waiter was rude but the desser   ['pasta', 'waiter', 'dessert']
Excellent ambiance and the wine selection is superb, though the   ['ambiance', 'wine selection', 'wait time']
The grilled chicken and the chocolate cake were both outstandin   ['grilled chicken', 'chocolate cake']
Not only was the food outstanding, but the little perks were gr   ['food', 'perks']
The bread is top notch as well.                                   ['bread']


In [15]:
# CELL 15 — Validate on test sentences that have known labels from CSV
# Pick 10 random test samples and check how well extraction works
print('📊 VALIDATION ON KNOWN TEST SAMPLES\n')
rng = random.Random(99)
sample_tests = rng.sample(te_samples, min(10, len(te_samples)))
correct = 0; total = 0
for (sid, text), true_aspects in sample_tests:
    true_terms = set(t.lower() for (t, _, _) in true_aspects)
    pred_terms = set(a.lower() for a in extract_aspects(text, best_model, tokenizer, DEVICE))
    exact_match = true_terms == pred_terms
    if exact_match: correct += 1
    total += 1
    print(f'Text    : "{text}"')
    print(f'True    : {sorted(true_terms)}')
    print(f'Pred    : {sorted(pred_terms)}')
    print(f'Match   : {"✅" if exact_match else "❌"}')
    print()
print(f'Exact match rate: {correct}/{total} = {correct/total*100:.0f}%')

📊 VALIDATION ON KNOWN TEST SAMPLES

Text    : "Orsay, is without a doubt one of the best values for authentic French food in NYC."
True    : ['french food']
Pred    : ['french food']
Match   : ✅

Text    : "Owner must have coem on this website to give himself credit."
True    : ['owner']
Pred    : ['owner']
Match   : ✅

Text    : "Barbecued codfish was gorgeously moist - as if poached - yet the fabulous texture was let down by curiously bland seasoning - a spice rub might have overwhelmed, however herb mix or other sauce would have done much to enhance."
True    : ['barbecued codfish', 'herb mix', 'sauce', 'seasoning', 'spice rub', 'texture']
Pred    : ['barbecued codfish', 'herb mix', 'sauce', 'seasoning', 'spice rub', 'texture']
Match   : ✅

Text    : "The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or not."
True    : ['food', 'kitchen', 'menu']
Pred    : ['food', 'kitchen', 'menu']
Matc

In [16]:
# CELL 16 — Save everything
results = {
    'model': MODEL_NAME, 'task': 'Aspect Term Extraction (BIO)',
    'data': 'SemEval 2014 Restaurants 70/15/15',
    'epochs': EPOCHS, 'test_f1': round(t_f1,4),
    'test_pr': round(t_pr,4), 'test_rc': round(t_rc,4),
    'best_val_f1': round(best_f1,4), 'history': history
}
with open(os.path.join(SAVE_DIR,'nb0_results.json'),'w') as f:
    json.dump(results, f, indent=2)
with open(os.path.join(SAVE_DIR,'label_map.json'),'w') as f:
    json.dump({'label2id':LABEL2ID,'id2label':ID2LABEL}, f, indent=2)

# Also save model name so NB4 knows which tokenizer to reload
with open(os.path.join(SAVE_DIR,'model_name.txt'),'w') as f:
    f.write(MODEL_NAME)

print('✅ Saved to:', SAVE_DIR)
print(f'\n📊 FINAL SUMMARY')
print(f'  Test F1        : {t_f1:.4f}')
print(f'  Test Precision : {t_pr:.4f}')
print(f'  Test Recall    : {t_rc:.4f}')
print(f'  Best Val F1    : {best_f1:.4f}')
print(f'  Model saved at : {best_path}')
print('\n👉 Next: Run NB4 — it will load from:', best_path)

✅ Saved to: /content/drive/MyDrive/Final year project/THE FINAL PROBLEM/aspect_extractor

📊 FINAL SUMMARY
  Test F1        : 0.8671
  Test Precision : 0.8548
  Test Recall    : 0.8797
  Best Val F1    : 0.9029
  Model saved at : /content/drive/MyDrive/Final year project/THE FINAL PROBLEM/aspect_extractor/best_model

👉 Next: Run NB4 — it will load from: /content/drive/MyDrive/Final year project/THE FINAL PROBLEM/aspect_extractor/best_model
